# Fintech Credit Risk Analysis
### Exposure Index — Financial Exposure Risk

**Notebook:** 03_exposure_analysis.ipynb  
**Phase:** 3 — Financial Exposure Risk  
**Input:** data/processed/cs_training_risk_scored.csv  
**Author:** Taofeek Salami  
**Date:** 2026-07-14

**Methodology note:** The dataset contains no loan balance or credit
limit fields, so exact financial exposure cannot be calculated. This
notebook builds an Exposure Index — a 0-6 composite score combining
utilization and debt ratio — as a relative, pre-default risk-ranking
proxy, not a dollar-value estimate.

Two data quality issues confirmed in Phase 3 SQL segmentation directly
affect this index and are handled explicitly rather than blended in:
1. `RevolvingUtilizationOfUnsecuredLines` defaults to a placeholder value
   (0.9999999) for 99.5% of borrowers with 0 open credit lines.
2. `DebtRatio` is not a reliable ratio for the 19.82% of borrowers with
   missing `MonthlyIncome` (90.04% of them show DebtRatio > 10).

In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/cs_training_risk_scored.csv')
print(df.shape)

(149999, 21)


## Component 1: Utilization Score (0-3)

Based on the bands validated in Phase 3 SQL (sql/01c_revenue_leakage.sql).
Borrowers with the confirmed placeholder value (0.9999999, occurring when
NumberOfOpenCreditLinesAndLoans = 0) are excluded from the standard
banding and instead scored using their actual credit-line risk behavior,
since the placeholder utilization value is not genuine.

In [2]:
def utilization_score(row):
    # Handle the confirmed placeholder: 0 open lines -> utilization
    # value is not genuine, so score based on the "0 lines" segment's
    # own validated risk level instead (25.64% default rate — highest
    # of any credit-line band, scored as high risk = 3)
    if row['NumberOfOpenCreditLinesAndLoans'] == 0:
        return 3

    util = row['RevolvingUtilizationOfUnsecuredLines']
    if util > 1:
        return 3  # Invalid/extreme, already validated as highest risk (37.25%)
    elif util >= 0.8:
        return 3  # High
    elif util >= 0.5:
        return 2  # Elevated
    elif util >= 0.3:
        return 1  # Moderate
    else:
        return 0  # Low

df['utilization_score'] = df.apply(utilization_score, axis=1)
print(df['utilization_score'].value_counts().sort_index())

utilization_score
0    92882
1    15830
2    16155
3    25132
Name: count, dtype: int64


## Component 2: Debt Ratio Score (0-3, income-valid borrowers only)

Only calculated for borrowers with valid MonthlyIncome
(income_missing = False). For the 19.82% with missing income, DebtRatio
is not a reliable ratio (confirmed in Phase 3 SQL) — the component is
left as null/NaN rather than fabricated, and these borrowers are flagged
as "exposure indeterminate" instead.

In [3]:
def debt_ratio_score(row):
    if row['income_missing']:
        return None  # Not reliable — handled separately, not fabricated

    dr = row['DebtRatio']
    if dr >= 1:
        return 3  # Overexposed
    elif dr >= 0.5:
        return 2  # High
    elif dr >= 0.3:
        return 1  # Moderate
    else:
        return 0  # Low

df['debt_ratio_score'] = df.apply(debt_ratio_score, axis=1)
print(df['debt_ratio_score'].value_counts(dropna=False).sort_index())

debt_ratio_score
0.0    60938
1.0    31141
2.0    20932
3.0     7257
NaN    29731
Name: count, dtype: int64


## Final Exposure Index (0-6)

Sum of utilization_score and debt_ratio_score. For borrowers with missing
income (debt_ratio_score = NaN), the index is left null and the borrower
is instead labeled "Exposure Indeterminate" — a distinct category, not
folded into a lower score. Faking a 0 for the missing component would
understate their real exposure; treating them as a separate group keeps
the index honest about its own limits.

In [4]:
df['exposure_index'] = df['utilization_score'] + df['debt_ratio_score']

def exposure_tier(row):
    if pd.isna(row['exposure_index']):
        return 'Indeterminate (income missing)'
    elif row['exposure_index'] <= 1:
        return 'Low'
    elif row['exposure_index'] <= 3:
        return 'Moderate'
    elif row['exposure_index'] <= 4:
        return 'High'
    else:
        return 'Critical'

df['exposure_tier'] = df.apply(exposure_tier, axis=1)
print(df['exposure_tier'].value_counts())

exposure_tier
Low                               63709
Moderate                          40394
Indeterminate (income missing)    29731
High                               9562
Critical                           6603
Name: count, dtype: int64


In [5]:
# Validate: does default rate increase cleanly across exposure tiers?
exposure_check = df.groupby('exposure_tier')['SeriousDlqin2yrs'].agg(['count', 'mean'])
print(exposure_check.sort_values('mean'))

                                count      mean
exposure_tier                                  
Low                             63709  0.023529
Indeterminate (income missing)  29731  0.056137
Moderate                        40394  0.097316
High                             9562  0.151746
Critical                         6603  0.223535


In [6]:
# Cross-tab: Exposure Index (pre-default signal) vs. Risk Tier (behavioral history)
crosstab = df.groupby(['risk_tier', 'exposure_tier'])['SeriousDlqin2yrs'].agg(['count', 'mean'])
print(crosstab)

                                          count      mean
risk_tier exposure_tier                                  
Critical  Critical                         1137  0.549692
          High                              837  0.519713
          Indeterminate (income missing)   2010  0.405473
          Low                               240  0.412500
          Moderate                         1132  0.484982
High      Critical                         1658  0.282268
          High                             2318  0.223469
          Indeterminate (income missing)   2859  0.131515
          Low                              3621  0.134217
          Moderate                         7558  0.221752
Low       High                             1002  0.068862
          Indeterminate (income missing)   1378  0.026851
          Low                             47155  0.011218
          Moderate                        11043  0.065019
Medium    Critical                         3808  0.100578
          High

In [7]:
# Recreate age bands (same boundaries used in Phase 2's SQL age-band
# segmentation) so Exposure Index findings can be cross-referenced
# against the age-based default patterns already established in Pillar 1
age_bins = pd.cut(df['age'], bins=[0, 29, 39, 49, 59, 120], labels=['20s', '30s', '40s', '50s', '60+'])
df['age_band'] = age_bins

# Cross-tab: does exposure tier separate risk consistently within each
# age band, or does age alone explain the pattern seen so far?
exposure_by_age = df.groupby(['age_band', 'exposure_tier'], observed=False)['SeriousDlqin2yrs'].agg(['count', 'mean'])
print(exposure_by_age)

                                         count      mean
age_band exposure_tier                                  
20s      Critical                          383  0.182768
         High                              440  0.190909
         Indeterminate (income missing)   1442  0.146325
         Low                              3096  0.048773
         Moderate                         3459  0.150043
30s      Critical                         1367  0.255304
         High                             1952  0.172643
         Indeterminate (income missing)   3123  0.114313
         Low                              8506  0.030802
         Moderate                         8235  0.125076
40s      Critical                         2049  0.245974
         High                             2939  0.163661
         Indeterminate (income missing)   5268  0.082194
         Low                             13527  0.027944
         Moderate                        10594  0.102133
50s      Critical              

## Feature Engineering — Income Quartile

### Create Income Quartile

To support borrower segmentation and interactive Tableau filtering, monthly income is divided into four equally sized groups (quartiles).

Borrowers with missing income values are assigned to a separate **"Missing"** category rather than being excluded. This preserves the full borrower population while making missing income an explicit analytical segment.

The resulting `income_quartile` feature is exported to the processed dataset and reused throughout the SQL analysis and Tableau dashboards.

In [9]:
# ==========================================================
# FEATURE ENGINEERING — INCOME QUARTILE
# ----------------------------------------------------------
# Divide borrowers into four income quartiles using
# MonthlyIncome. Missing income values are retained as
# a separate category to preserve the full population
# for downstream analysis and Tableau filtering.
# ==========================================================

df["income_quartile"] = pd.qcut(
    df["MonthlyIncome"],
    q=4,
    labels=["Q1 (Lowest)", "Q2", "Q3", "Q4 (Highest)"]
)

df["income_quartile"] = (
    df["income_quartile"]
      .cat.add_categories(["Missing"])
      .fillna("Missing")
)

In [10]:
print(df["income_quartile"].value_counts(dropna=False))

income_quartile
Q1 (Lowest)     30289
Q4 (Highest)    30059
Q2              30026
Q3              29894
Missing         29731
Name: count, dtype: int64


In [11]:
df.to_csv('../data/processed/cs_training_exposure_scored.csv', index=False)
print(df.shape)

(149999, 27)
